#BERT

허깅 페이스 트랜스포머스 라이브러리의 BERT모델과 네이버 영화리뷰 감정분석 데이터세트를 활용해 분류 모델 학습

In [ ]:
!pip install Korpora

In [ ]:
import numpy as np
import pandas as pd
from Korpora import Korpora

corpus = Korpora.load("nsmc")
df = pd.DataFrame(corpus.test).sample(20000, random_state=42)
train, valid, test = np.split(
    df.sample(frac=1, random_state=42), [int(0.6 * len(df)), int(0.8 * len(df))]
)

print(train.head(5).to_markdown())
print(f"Training Data Size : {len(train)}")
print(f"Validation Data Size : {len(valid)}")
print(f"Test Data Size : {len(test)}")


    Korpora 는 다른 분들이 연구 목적으로 공유해주신 말뭉치들을
    손쉽게 다운로드, 사용할 수 있는 기능만을 제공합니다.

    말뭉치들을 공유해 주신 분들에게 감사드리며, 각 말뭉치 별 설명과 라이센스를 공유 드립니다.
    해당 말뭉치에 대해 자세히 알고 싶으신 분은 아래의 description 을 참고,
    해당 말뭉치를 연구/상용의 목적으로 이용하실 때에는 아래의 라이센스를 참고해 주시기 바랍니다.

    # Description
    Author : e9t@github
    Repository : https://github.com/e9t/nsmc
    References : www.lucypark.kr/docs/2015-pyconkr/#39

    Naver sentiment movie corpus v1.0
    This is a movie review dataset in the Korean language.
    Reviews were scraped from Naver Movies.

    The dataset construction is based on the method noted in
    [Large movie review dataset][^1] from Maas et al., 2011.

    [^1]: http://ai.stanford.edu/~amaas/data/sentiment/

    # License
    CC0 1.0 Universal (CC0 1.0) Public Domain Dedication
    Details in https://creativecommons.org/publicdomain/zero/1.0/

[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ratings_train.txt
[Korpora] Corpus `nsmc` is already installed at /root/Korpora/nsmc/ra

/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


BERT토크나이저(BertTokenizer)클래스로 데이터를 전처리 -> BERT토크나이저 클래스로 데이터를 전처리하고 데이터로더에 적용

In [ ]:
import torch
from transformers import BertTokenizer
from torch.utils.data import TensorDataset, DataLoader
from torch.utils. data import RandomSampler, SequentialSampler

def make_dataset(data, tokenizer, device):
  tokenized = tokenizer(
      text=data.text.tolist(),
      padding="longest",
      truncation=True,
      return_tensors="pt"
  )
  input_idx = tokenized["input_ids"].to(device)
  attention_mask = tokenized["attention_mask"].to(device)
  labels = torch.tensor(data.label.values, dtype=torch.long).to(device)
  return TensorDataset(input_idx, attention_mask, labels)

def get_dataloader(dataset, sampler, batch_size):
  data_sampler = sampler(dataset)
  dataloader = DataLoader(dataset, sampler=data_sampler, batch_size=batch_size)
  return dataloader

epochs=5
batch_size=32
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = BertTokenizer.from_pretrained(
    pretrained_model_name_or_path="bert-base-multilingual-cased",
    do_lower_case=False
)

train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)

valide_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(valide_dataset, SequentialSampler, batch_size)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size)

print(train_dataset[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


(tensor([   101,  58466,   9812, 118956, 119122,  59095,  10892,   9434, 118888,
           117,   9992,  40032,  30005,    117,   9612,  37824,   9410,  12030,
         42337,  10739,  83491,  12508,    106,    106,    102,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,      0,      0,      0,      0,      0,
             0,      0,      0,      0,

사전 학습된 모델은bert-base-multilingual-cased로 다중언어를 지월, 대소문자를 유지하는 사전학습된 BERT 모델을 의미. 소문자 유지(do_Lower_case)매개변수를 False로 할당해 소문자로 변환하지 않게함( apple != APPLE)
- get_dataloader함수: 샘플러 클래스를 활용해 데이터를 목적에 따라 샘플림
- 무작위 샘플러(RandomSampler)무작위 샘플링
- 시퀀셜샘플러: 데이터를 고정된 순서대로 반환->검증/배치에 적용

In [ ]:
#모델 및 최적화 함수 선언
from torch import optim
from transformers import BertForSequenceClassification

model = BertForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="bert-base-multilingual-cased",
    num_labels=2
).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(BERT문장 분류 모델)로 버트 모델 불러옴, 패칭 기법 사용하므로 모델 설정 변경x
AdamW: Adam최적화 함수에 가중치 감쇠를 추가한 변형된 경사 하강법 알고리즘.
  - 안정적인 기울기 갱신, 빠르게 수렴

In [ ]:
#모델 구조 확인

model = BertForSequenceClassification.from_pretrained(pretrained_model_name_or_path="bert-base-multilingual-cased")


for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("└", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("│  └", ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("│  │  └", sssub_name)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


bert
└ embeddings
│  └ word_embeddings
│  └ position_embeddings
│  └ token_type_embeddings
│  └ LayerNorm
│  └ dropout
└ encoder
│  └ layer
│  │  └ 0
│  │  └ 1
│  │  └ 2
│  │  └ 3
│  │  └ 4
│  │  └ 5
│  │  └ 6
│  │  └ 7
│  │  └ 8
│  │  └ 9
│  │  └ 10
│  │  └ 11
└ pooler
│  └ dense
│  └ activation
dropout
classifier


In [ ]:
#모델 학습 및 평가
import numpy as np
from torch import nn
from torch import optim

model = BertForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="bert-base-multilingual-cased",
    num_labels=2
).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)

def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for input_ids, attention_mask, labels in dataloader:
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss = train_loss / len(dataloader)
    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        criterion = nn.CrossEntropyLoss()
        val_loss, val_accuracy = 0.0, 0.0

        for input_ids, attention_mask, labels in dataloader:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            logits = outputs.logits

            loss = criterion(logits, labels)
            logits = logits.detach().cpu().numpy()
            label_ids = labels.to("cpu").numpy()
            accuracy = calc_accuracy(logits, label_ids)

            val_loss += loss
            val_accuracy += accuracy

    val_loss = val_loss/len(dataloader)
    val_accuracy = val_accuracy/len(dataloader)
    return val_loss, val_accuracy


best_loss = 10000
for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss, val_accuracy = evaluation(model, valid_dataloader)
    print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f} Val Accuracy {val_accuracy:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "BertForSequenceClassification.pt")
        print("Saved the model weights")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
model = BertForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="bert-base-multilingual-cased",
    num_labels=2
).to(device)
model.config.pad_token_id = model.config.eos_token_id
model.load_state_dict(torch.load("BertForSequenceClassification.pt"))

test_loss, test_accuracy = evaluation(model, test_dataloader)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

#BART

In [1]:
pip install datasets

In [1]:
import numpy as np
from datasets import load_dataset

news = load_dataset("argilla/news-summary", split= "test")
df = news.to_pandas().sample(5000, random_state=42)[["text", "prediction"]]
df["prediction"] = df["prediction"].map(lambda x: x[0]["text"])
train, valid, test = np.split(
    df.sample(frac=1, random_state=42), [int(0.6 * len(df)), int(0.8 * len(df))]
)

print(f"Source News: {train.text.iloc[0][:200]}")
print(f"Summerization : {train.prediction.iloc[0][:50]}")
print(f"Training Data Size: {len(train)}")
print(f"Validation Data Size: {len(valid)}")
print(f"Test Data Size: {len(test)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/2.02k [00:00<?, ?B/s]

data/train-00000-of-00001-ebc48879f34571(…):   0%|          | 0.00/1.54M [00:00<?, ?B/s]

data/test-00000-of-00001-6227bd8eb10a9b5(…):   0%|          | 0.00/31.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/20417 [00:00<?, ? examples/s]

Source News: DANANG, Vietnam (Reuters) - Russian President Vladimir Putin said on Saturday he had a normal dialogue with U.S. leader Donald Trump at a summit in Vietnam, and described Trump as civil, well-educated
Summerization : Putin says had useful interaction with Trump at Vi
Training Data Size: 3000
Validation Data Size: 1000
Test Data Size: 1000


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


In [2]:
import torch
from transformers import BartTokenizer
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import RandomSampler, SequentialSampler
from torch.nn.utils.rnn import pad_sequence


def make_dataset(data, tokenizer, device):
    tokenized = tokenizer(
        text=data.text.astype(str).tolist(),
        padding="longest",
        truncation=True,
        max_length=512,
        return_tensors="pt"
    )

    input_ids = tokenized["input_ids"].to(device)
    attention_mask = tokenized["attention_mask"].to(device)

    labels = []

    for target in data.prediction:
        encoded_label = tokenizer.encode(
            str(target),
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).squeeze()

        labels.append(encoded_label)

    labels = pad_sequence(
        labels,
        batch_first=True,
        padding_value=-100
    ).to(device)

    return TensorDataset(input_ids, attention_mask, labels)


def get_dataloader(dataset, sampler, batch_size):
    data_sampler = sampler(dataset)

    dataloader = DataLoader(
        dataset,
        sampler=data_sampler,
        batch_size=batch_size
    )

    return dataloader


epochs = 3
batch_size = 8
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = BartTokenizer.from_pretrained(
    "facebook/bart-base"
)

train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(
    train_dataset,
    RandomSampler,
    batch_size
)

valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(
    valid_dataset,
    SequentialSampler,
    batch_size
)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(
    test_dataset,
    SequentialSampler,
    batch_size
)

print(train_dataset[0])

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

(tensor([    0,   495,  1889,  9298,     6,  5490,    36,  1251,    43,   111,
         1083,   270,  6546,  3176,    26,    15,   378,    37,    56,    10,
         2340,  6054,    19,   121,     4,   104,     4,   884,   807,   140,
           23,    10,  3564,    11,  5490,     6,     8,  1602,   140,    25,
         2366,     6,   157,    12, 26414,     6,     8,  3473,     7,   432,
           19,     4,  3176,    26,    14,    10, 30036,   196,  9526,  2662,
           12,  3955,   529,    19,   140,   222,    45,  1369,    23,     5,
         1817,    12,  8145,  4713, 18204,  3564,     6,  4319, 19114,   743,
           15,   258,  2380,     8, 20022, 11883,   743,     4,  3176,     6,
           23,    10,  7515,    13,  1865,    23,     5,   253,     9,     5,
         3564,     6,    26,    89,    21,   202,    10,   240,    13,   617,
          121,     4,   104,  3358, 15685,  9872,     6,   258,    23,     5,
          672,     9,  3885,     9,   194,     8,    49,   503,

In [3]:
from torch import optim
from transformers import BartForConditionalGeneration

model = BartForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path="facebook/bart-base"
).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)

config.json:   0%|          | 0.00/1.72k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [4]:
#모델 구조 확인

model = BartForConditionalGeneration.from_pretrained(pretrained_model_name_or_path="facebook/bart-base")


for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("└", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("│  └", ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("│  │  └", sssub_name)

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

model
└ shared
└ encoder
│  └ embed_tokens
│  └ embed_positions
│  └ layers
│  │  └ 0
│  │  └ 1
│  │  └ 2
│  │  └ 3
│  │  └ 4
│  │  └ 5
│  └ layernorm_embedding
└ decoder
│  └ embed_tokens
│  └ embed_positions
│  └ layers
│  │  └ 0
│  │  └ 1
│  │  └ 2
│  │  └ 3
│  │  └ 4
│  │  └ 5
│  └ layernorm_embedding
lm_head


In [6]:
pip install evaluate rouge_score absl-py

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.4 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=69d1d0307a76c31e8aa9e5b681c132de47619251ce1e405abb167934e19bad48
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
import numpy as np
import evaluate
from transformers import BartForConditionalGeneration
from torch import optim

model = BartForConditionalGeneration.from_pretrained(
    "facebook/bart-base"
).to(device)

optimizer = optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)
rouge_score = evaluate.load("rouge")

def calc_rouge(pred_ids, labels):
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(
        pred_ids,
        skip_special_tokens=True
    )

    decoded_labels = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True
    )

    rouge = rouge_score.compute(
        predictions=decoded_preds,
        references=decoded_labels
    )

    return rouge["rouge2"]


def train_epoch(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for input_ids, attention_mask, labels in dataloader:
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    return train_loss / len(dataloader)


def evaluation(model, dataloader):
    model.eval()
    val_loss = 0.0
    val_rouge = 0.0

    with torch.no_grad():
        for input_ids, attention_mask, labels in dataloader:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            loss = outputs.loss

            generated_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_length=128
            )

            pred_ids = generated_ids.detach().cpu().numpy()
            label_ids = labels.detach().cpu().numpy()

            rouge = calc_rouge(pred_ids, label_ids)

            val_loss += loss.item()
            val_rouge += rouge

    return val_loss / len(dataloader), val_rouge / len(dataloader)


best_loss = 10000

for epoch in range(epochs):
    train_loss = train_epoch(model, optimizer, train_dataloader)
    val_loss, val_rouge = evaluation(model, valid_dataloader)

    print(
        f"Epoch {epoch + 1}: "
        f"Train Loss: {train_loss:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Rouge2: {val_rouge:.4f}"
    )

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "BartForConditionalGeneration.pt")
        print("Saved the model weights")

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

In [ ]:
model = BartForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path="facebook/bart-base"
).to(device)
model.load_state_dict(torch.load("../models/BartForConditionalGeneration.pt"))

test_loss, test_rouge = evaluation(model, test_dataloader)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Rouge2: {test_rouge:.4f}")

In [ ]:
from transformers import pipeline

summerizer = pipeline(
    task="summarization",
    model=model,
    tokenizer=tokenizer
    max_length=54,
    device="cpu"
)

for index in range(5):
  news_text = test.text.iloc[index]
  summerization = test.prediction.iloc[index]
  predicted_summerization = summerizer(news_text)[0]["summary_text"]
  print(f"정답 요약문: {summerization}")
  print(f"예측 요약문: {predicted_summerization}\n")

#ELECTRA

In [ ]:
import torch
from transformers import ElectraTokenizer
from torch.utils.data import TensorDataset, DataLoader
from torch.utils. data import RandomSampler, SequentialSampler

def make_dataset(data, tokenizer, device):
  tokenized = tokenizer(
      text=data.text.tolist(),
      padding="longest",
      truncation=True,
      return_tensors="pt"
  )
  input_idx = tokenized["input_ids"].to(device)
  attention_mask = tokenized["attention_mask"].to(device)
  labels = torch.tensor(data.label.values, dtype=torch.long).to(device)
  return TensorDataset(input_idx, attention_mask, labels)

def get_dataloader(dataset, sampler, batch_size):
  data_sampler = sampler(dataset)
  dataloader = DataLoader(dataset, sampler=data_sampler, batch_size=batch_size)
  return dataloader

epochs=5
batch_size=32
device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = ElectraTokenizer.from_pretrained(
    pretrained_model_name_or_path="monologg/koelectra-base-v3-discriminator",
    do_lower_case=False
)

train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(train_dataset, RandomSampler, batch_size)

valide_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(valide_dataset, SequentialSampler, batch_size)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(test_dataset, SequentialSampler, batch_size)

print(train_dataset[0])

In [ ]:
from torch import optim
from transformers import ElectraForSequenceClassification

model = ElectraForSequenceClassification,from_pretrained(
    pretrained_model_name_or_path="monologg/koelectra-base-v3-discriminator",
    num_labels=2
).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)

In [ ]:
#모델 구조 확인

model = ElectraForSequenceClassification.from_pretrained(pretrained_model_name_or_path="monologg/koelectra-base-v3-discriminator")


for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("└", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("│  └", ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("│  │  └", sssub_name)

In [ ]:
#모델 학습 및 평가
import numpy as np
from torch import nn
from torch import optim

model = ElectraForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="monologg/koelectra-base-v3-discriminator",
    num_labels=2
).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)

def calc_accuracy(preds, labels):
    pred_flat = np.argmax(preds, axis=1).flatten()
    labels_flat = labels.flatten()
    return np.sum(pred_flat == labels_flat) / len(labels_flat)

def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for input_ids, attention_mask, labels in dataloader:
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss = train_loss / len(dataloader)
    return train_loss

def evaluation(model, dataloader):
    with torch.no_grad():
        model.eval()
        criterion = nn.CrossEntropyLoss()
        val_loss, val_accuracy = 0.0, 0.0

        for input_ids, attention_mask, labels in dataloader:
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            logits = outputs.logits

            loss = criterion(logits, labels)
            logits = logits.detach().cpu().numpy()
            label_ids = labels.to("cpu").numpy()
            accuracy = calc_accuracy(logits, label_ids)

            val_loss += loss
            val_accuracy += accuracy

    val_loss = val_loss/len(dataloader)
    val_accuracy = val_accuracy/len(dataloader)
    return val_loss, val_accuracy


best_loss = 10000
for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss, val_accuracy = evaluation(model, valid_dataloader)
    print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f} Val Loss: {val_loss:.4f} Val Accuracy {val_accuracy:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(model.state_dict(), "ElectraForSequenceClassification.pt")
        print("Saved the model weights")

In [ ]:
model = ElectraForSequenceClassification.from_pretrained(
    pretrained_model_name_or_path="electra-base-multilingual-cased",
    num_labels=2
).to(device)
model.config.pad_token_id = model.config.eos_token_id
model.load_state_dict(torch.load("ElectraForSequenceClassification.pt"))

test_loss, test_accuracy = evaluation(model, test_dataloader)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

#T5

In [ ]:
import numpy as np
from datasets import load_dataset

news = load_dataset("argilla/news-summary", split="test")
df = news.to_pandas().sample(5000, random_state=42)[["text", "prediction"]]
df["text"] = "summerize: "+ df["text"]
df["prediction"] = df["prediction"].map(lambda x: x[0]["text"])
train, valid, test = np.split(
    df.sample(frac=1, random_state=42), [int(0.6 * len(df)), int(0.8 * len(df))]
)

print(f"Source News : {train.text.o;pc[0][:200]}")
print(f"Summerization : {train.prediction.iloc[0][:50]}")

In [ ]:
import torch
from transformers import T5Tokenizer
from torch.utils.data import TensorDataset, DataLoader
from torch.utils.data import RandomSampler, Sequential Sampler

def make_dataset(data, tokenizer, device):
  source = tokenizer(
      text=dat.text.tolist(),
      padding="max_length",
      max_length=128,
      pad_to_max_length=True,
      truncation=True,
      return_tensors="pt"
  )

  target = tokenizer(
    text=data.prediction.tolist(),
    padding="max_length",
    max_length=128,
    pad_to_max_length=True,
    truncation=True,
    return_tensors="pt"
  )
  source_ids = source["input_ids"].squeeze().to(device)
  source_mask = source["attention_mask"].squeeze().to(device)
  target_ids = target["input_ids"].squeeze().to(device)
  target_mask = target["attention_mask"].squeeze().to(device)
  return TensorDataset(source_ids, source_mask, target_ids, target_mask)

epochs=3
batch_size=8
device="cuda" if torch.cuda.is_available() else "cpu"
tokenizer=T5Tokenizer.from_pretrained(
    pretrained_model_name_or_path="t5-small"
)


train_dataset = make_dataset(train, tokenizer, device)
train_dataloader = get_dataloader(
    train_dataset,
    RandomSampler,
    batch_size
)

valid_dataset = make_dataset(valid, tokenizer, device)
valid_dataloader = get_dataloader(
    valid_dataset,
    SequentialSampler,
    batch_size
)

test_dataset = make_dataset(test, tokenizer, device)
test_dataloader = get_dataloader(
    test_dataset,
    SequentialSampler,
    batch_size
)

print(next(iter(train_dataloader)))
print(tokenizer.convert_ids_to_tokens(21603))
print(tokenizer.convert_ids_to_tokens(10))


In [ ]:
from torch import optim
from transformers import T5ForConditionalGeneration

model = T5ForConditionalGeneration.from_pretrained(
    pretrained_model_name_or_path="t5-small"
).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-5, eps=1e-8)


In [ ]:

model = T5ForConditionalGeneration.from_pretrained(pretrained_model_name_or_path="t5-small")


for main_name, main_module in model.named_children():
    print(main_name)
    for sub_name, sub_module in main_module.named_children():
        print("└", sub_name)
        for ssub_name, ssub_module in sub_module.named_children():
            print("│  └", ssub_name)
            for sssub_name, sssub_module in ssub_module.named_children():
                print("│  │  └", sssub_name)

In [ ]:
import torch
import numpy as np
from torch import optim

def train(model, optimizer, dataloader):
    model.train()
    train_loss = 0.0

    for source_ids, source_mask, target_ids, target_mask in dataloader:
        decoder_input_ids = target_ids[:, :-1].contiguous()

        labels = target_ids[:, 1:].clone().detach()
        labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100

        outputs = model(
            input_ids=source_ids,
            attention_mask=source_mask,
            decoder_input_ids=decoder_input_ids,
            labels=labels
        )

        loss = outputs.loss
        train_loss += loss.item()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    train_loss = train_loss / len(dataloader)
    return train_loss


def evaluation(model, dataloader):
    model.eval()
    val_loss = 0.0

    with torch.no_grad():
        for source_ids, source_mask, target_ids, target_mask in dataloader:
            decoder_input_ids = target_ids[:, :-1].contiguous()

            labels = target_ids[:, 1:].clone().detach()
            labels[target_ids[:, 1:] == tokenizer.pad_token_id] = -100

            outputs = model(
                input_ids=source_ids,
                attention_mask=source_mask,
                decoder_input_ids=decoder_input_ids,
                labels=labels
            )

            loss = outputs.loss
            val_loss += loss.item()

    val_loss = val_loss / len(dataloader)
    return val_loss


best_loss = 10000

for epoch in range(epochs):
    train_loss = train(model, optimizer, train_dataloader)
    val_loss = evaluation(model, valid_dataloader)

    print(
        f"Epoch {epoch + 1}: "
        f"Train Loss: {train_loss:.4f} "
        f"Val Loss: {val_loss:.4f}"
    )

    if val_loss < best_loss:
        best_loss = val_loss
        torch.save(
            model.state_dict(),
            "../models/T5ForConditionalGeneration.pt"
        )
        print("Saved the model weights")

In [ ]:
model.eval()

with torch.no_grad():
    for input_ids, attention_mask, labels in test_dataloader:
        generated_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_length=128,
            num_beams=3,
            repetition_penalty=2.5,
            length_penalty=1.0,
            early_stopping=True
        )

        for generated, target in zip(generated_ids, labels):
            pred = tokenizer.decode(
                generated,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True
            )

            target = target.clone()
            target[target == -100] = tokenizer.pad_token_id

            actual = tokenizer.decode(
                target,
                skip_special_tokens=True,
                clean_up_tokenization_spaces=True
            )

            print("Generated Headline Text:", pred)
            print("Actual Headline Text:", actual)
            print()

        break